In [9]:
from pathlib import Path
import polars as pl

def load_data(file_path: Path | str, lazy: bool = False) -> pl.DataFrame | pl.LazyFrame | None:
    """
    Load data from a CSV file into a Polars DataFrame or LazyFrame.

    Parameters:
    file_path (Path | str): Absolute or relative path to the CSV file.
    lazy (bool): If True, returns a LazyFrame for deferred execution.

    Returns:
    pl.DataFrame | pl.LazyFrame | None: Loaded data, or None if reading fails.
    """
    try:
        if lazy:
            # Scans metadata without reading all rows into memory immediately
            return pl.scan_csv(file_path)
        return pl.read_csv(file_path)
    except Exception as e:
        print(f"Error loading data: {e}")
        return None



In [ ]:
ROOT_DIR = Path.cwd().parents[1]
file_path = ROOT_DIR / "instructions" / "applications_dataset_1.csv"

df = load_data(file_path)

if df is not None:
    print("Schema:")
    print(df.schema)
    print("\nFirst 5 rows:")
    print(df.head(5))

Schema:
Schema({'name': String, 'email': String, 'date_of_birth': String, 'mobile_no': String})

First 5 rows:
shape: (5, 4)
┌────────────────┬─────────────────────────────────┬───────────────┬───────────┐
│ name           ┆ email                           ┆ date_of_birth ┆ mobile_no │
│ ---            ┆ ---                             ┆ ---           ┆ ---       │
│ str            ┆ str                             ┆ str           ┆ str       │
╞════════════════╪═════════════════════════════════╪═══════════════╪═══════════╡
│ William Dixon  ┆ William_Dixon@woodward-fuller.… ┆ 1986/01/10    ┆ 40601711  │
│ Kristen Horn   ┆ Kristen_Horn@lin.com            ┆ 1974-09-10    ┆ 737931    │
│ Kimberly Chang ┆ Kimberly_Chang@johnson-lopez.b… ┆ 02/27/1974    ┆ 2692047   │
│ Mary Ball      ┆ Mary_Ball@stevens.biz           ┆ 02/05/1968    ┆ 886359    │
│ Benjamin Craig ┆ Benjamin_Craig@berry.net        ┆ 12/11/1988    ┆ 696429    │
└────────────────┴─────────────────────────────────┴─────────────

In [ ]:
import hashlib
import tempfile
from datetime import datetime
from pathlib import Path

import polars as pl

from membership_pipeline import config, ingest, transform, validate, output

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(38)

DEMO_BATCH = """name,email,date_of_birth,mobile_no
Mr. William Dixon,William_Dixon@provider.com,1986/01/10,4060 1711
Sherry Gonzalez DDS,Sherry_Gonzalez@caldwell.net,14-03-1973,66744895
Kristen Horn,Kristen_Horn@lin.com,1974-09-10,737931
Cathy Werner,Cathy_Werner@martinez.net,2018-09-25,71380411
Mary Ball,Mary_Ball@stevens.biz,02/05/1968,88635912
Lori Torres,Lori_Torres@hill.net,1996/02/31,57951752
,Nobody_Here@jackson.com,1990-01-01,12345678
Kimberly Chang,Kimberly_Chang@johnson-lopez.biz,02/27/1974,2692047
"""

DEMO_HOME = Path(tempfile.mkdtemp())
DEMO_INPUT = DEMO_HOME / "input_batches"
DEMO_INPUT.mkdir()
(DEMO_INPUT / "demo_batch.csv").write_text(DEMO_BATCH, encoding="utf-8")

files = ingest.discover_batches(DEMO_INPUT)
raw = ingest.scan_batches(files)

print(type(raw))
print(raw.collect_schema())
print(raw.drop("source_file").collect())

In [ ]:
cleaned = transform.clean(raw)
cleaned_df = cleaned.collect()

print("NAME ->")
print(cleaned_df.select("name", "first_name", "last_name"))

print("\nBIRTHDAY ->")
print(cleaned_df.select("birthday", "birthday_date", "birthday_ymd", "age", "above_18"))

print("\nMOBILE ->")
print(cleaned_df.select("mobile", "mobile_normalised"))

In [ ]:
print("FAILED RULES (True = this rule rejected the row)")
print(cleaned_df.select(
    pl.col("name"),
    **{reason: ~passes for reason, passes in validate.predicates()},
))

validated = validate.validate(cleaned)
validated_df = validated.collect()

print("\nVERDICT")
print(validated_df.select("name", "is_successful", "rejection_reason", "membership_id"))

In [ ]:
for name, last, ymd, mid in validated_df.filter(pl.col("is_successful")).select(
    "name", "last_name", "birthday_ymd", "membership_id"
).iter_rows():
    digest = hashlib.sha256(ymd.encode()).hexdigest()
    print(f"{name:<22} {last} + sha256({ymd})[:5] = {digest[:5]}  ->  {mid}")
    print(f"{'':<22} full digest: {digest}")

In [ ]:
config.configure_logging("INFO")

RUN_TS = datetime(2022, 3, 1, 9, 0, 0)
summary = output.write_outputs(validated, DEMO_HOME / "output", run_ts=RUN_TS)
archived = output.archive_batches(files, DEMO_HOME / "archive", RUN_TS)

print(f"\nrows_in={summary.rows_in} "
      f"successful={summary.successful} unsuccessful={summary.unsuccessful}")

for kind, path in summary.output_paths.items():
    print(f"\n{kind.upper()}  ->  {path.name}")
    print(pl.read_csv(path, infer_schema=False).drop("source_file", strict=False))

print("\narchived:", [p.name for p in archived])
print("input_batches now:", [p.name for p in DEMO_INPUT.glob("*.csv")])